In [0]:
%sql
CREATE CATALOG IF NOT EXISTS RETAIL_LAKEHOUSE;

In [0]:
%sql
USE CATALOG RETAIL_LAKEHOUSE;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS retail_lakehouse.bronze
comment 'raw data'

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS retail_lakehouse.silver
comment 'cleaned data';
    
CREATE SCHEMA IF NOT EXISTS retail_lakehouse.gold
comment 'aggregated data';

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS retail_lakehouse.bronze.landingzone
comment "Raw data multi source landing zone";

In [0]:
# /Volumes/retail_lakehouse/bronze/landingzone

In [0]:
from pyspark.sql import functions as F

In [0]:
volume_root="/Volumes/retail_lakehouse/bronze/landingzone"
checkpoint_root='/Volumes/retail_lakehouse/bronze/landingzone/checkpoints'


In [0]:
#json data ingestion 

order_json_path =f'{volume_root}/orders_stream'

orders_stream=(
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format","json")
    .option("cloudFiles.schemaLocation",f"{checkpoint_root}/orders_stream")
    .option("cloudFiles.inferColumnTypes",True)
    #orders_stream.json is a pretty-printed JSON array of nested order objects,
    #so it must be parsed with multiline, not line-by-line 
    .option("multiline",True)
    .option("pathGlobFilter","*.json")
    .load(order_json_path)
    .withColumn("_ingested_at",F.current_timestamp())
    .withColumn("_source_file",F.col("_metadata").getField("file_path"))
)

In [0]:
#trigger - execute incremental batchwise 
(
orders_stream.writeStream
.format("delta")
.outputMode("append")
.option("checkpointLocation",f"{checkpoint_root}/orders_delta")
.trigger(availableNow=True)
.option("mergeSchema",True)
.toTable("retail_lakehouse.bronze.orders_raw")
)

print ("orders json streaming ingestion completed")

In [0]:
customer_parquet_path=f'{volume_root}/customers.parquet'
customer_df=(
    spark.read.format("parquet")
    .load(customer_parquet_path)
    .withColumn("_ingested_at",F.current_timestamp())
    .withColumn("_source_file",F.col("_metadata").getField("file_path"))
)             
#trigger - execute incremental batchwise 

customer_df.write.format("delta").mode("append").saveAsTable("retail_lakehouse.bronze.customers_raw")

print ("customer parquet streaming ingestion completed")


In [0]:
products_csv_path=f'{volume_root}/products.csv'
products_df=(
    spark.read.format("csv")
    .option("header",True)
    .option("inferSchema",True)
    .load(products_csv_path)
    .withColumn("_ingested_at",F.current_timestamp())
    .withColumn("_source_file",F.col("_metadata").getField("file_path"))
)             
#trigger - execute incremental batchwise 

products_df.write.format("delta").mode("append").saveAsTable("retail_lakehouse.bronze.products_raw")

print ("products csv streaming ingestion completed")

*** silver layer ***

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

raw_orders = spark.table("retail_lakehouse.bronze.orders_raw")

In [0]:
# flatten json

flatten_orders = (
    raw_orders
    .withColumn("item", F.explode("items"))
    .select(
        F.col("order_id"),
        F.col("customer_id"),
        F.to_timestamp(F.col("order_timestamp")).alias("order_timestamp"),
        F.col("payment.method").alias("payment_method"),
        F.col("payment.status").alias("payment_status"),
        F.col("shipping_address.city").alias("shipping_city"),
        F.col("shipping_address.state").alias("shipping_state"),
        F.col("item.product_id").alias("product_id"),
        F.col("item.qty").alias("quantity"),
        F.col("item.unit_price").cast("double").alias("unit_price"),
        (F.col("item.qty") *
         F.col("item.unit_price")).alias("line_total_amount"),
        F.col("_ingested_at")
    )
)

In [0]:
# quality check (we will drop the corrupted rows)

valid_orders = (
    flatten_orders
    .filter(F.col("order_id").isNotNull() & (F.col("order_id")!=""))
    .filter(F.col("customer_id").isNotNull() & (F.col("customer_id")!=""))
    .filter(F.col("quantity") > 0)
    .filter(F.col("unit_price") > 0)
    .filter(F.col("payment_status") == "SUCCESS")
)

In [0]:
# dedpulication on composite key (order_id + product_id)

window_spec = Window.partitionBy("order_id", "product_id").orderBy(F.col("_ingested_at").desc())

deduplicated_orders = (
    valid_orders
    .withColumn("raw_rank", F.row_number().over(window_spec))
    .filter(F.col("raw_rank") == 1)
    .drop("raw_rank", "_ingested_at")
)

In [0]:
# idempotent upsert
target_silver_table = "retail_lakehouse.silver.orders_clean"

if not spark.catalog.tableExists(target_silver_table):
    deduplicated_orders.write.format("delta").saveAsTable(target_silver_table)
    print("Silver table Initiated")
else:
    silver_target = DeltaTable.forName(spark, target_silver_table)
    (
        silver_target.alias("target")
        .merge(
            deduplicated_orders.alias("source"),
            "target.order_id = source.order_id AND target.product_id = source.product_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    ) 
    print("composite delta merge completed here.....")

In [0]:
spark.table("retail_lakehouse.bronze.products_raw").dropDuplicates(["product_id"]).write.format("delta").mode("overwrite").saveAsTable("retail_lakehouse.silver.products_clean")

spark.table("retail_lakehouse.bronze.customers_raw").dropDuplicates(["customer_id"]).write.format("delta").mode("overwrite").saveAsTable("retail_lakehouse.silver.customers_clean")
     